In [1]:
import os
import time
 
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, from_json, from_unixtime, to_date
from pyspark.sql.types import (
    DoubleType,
    IntegerType,
    LongType,
    StructField,
    StructType,
)
 
KAFKA_BOOTSTRAP_SERVERS = "urbangreen-kafka:9092"
KAFKA_TOPIC = "sensor_readings"
 
MINIO_ENDPOINT = "http://urbangreen-minio:9000"
MINIO_ACCESS_KEY = os.getenv("MINIO_ROOT_USER", "minioadmin")
MINIO_SECRET_KEY = os.getenv("MINIO_ROOT_PASSWORD", "minioadmin123")
MINIO_BUCKET = os.getenv("MINIO_STAGING_BUCKET", "staging")
 
spark = (
    SparkSession.builder
    .appName("sensor-readings-stream-notebook-test")
    .master("local[*]")
    .config("spark.driver.host", "urbangreen-jupyter")
    .config("spark.driver.bindAddress", "0.0.0.0")
    .config("spark.driver.port", "7078")
    .config("spark.blockManager.port", "7079")
    .config("spark.executor.memory", "512m")
    .config("spark.executor.cores", "1")
    .config("spark.cores.max", "2")

 
    # Kafka connector
    .config(
        "spark.jars.packages",
        ",".join([
            "org.apache.spark:spark-sql-kafka-0-10_2.13:4.0.2",
            "org.apache.hadoop:hadoop-aws:3.4.1",
            "com.clickhouse:clickhouse-jdbc:0.9.8",
        ])
    )
 
    # MinIO/S3A config
    .config("spark.hadoop.fs.s3a.endpoint", MINIO_ENDPOINT)
    .config("spark.hadoop.fs.s3a.access.key", MINIO_ACCESS_KEY)
    .config("spark.hadoop.fs.s3a.secret.key", MINIO_SECRET_KEY)
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
    .config(
        "spark.hadoop.fs.s3a.aws.credentials.provider",
        "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider",
    )
    .getOrCreate()
)
 
spark.sparkContext.setLogLevel("WARN")
 
print("Spark version:", spark.version)
print("MinIO endpoint:", MINIO_ENDPOINT)
print("Bucket:", MINIO_BUCKET)

:: loading settings :: url = jar:file:/opt/conda/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /home/jovyan/.ivy2.5.2/cache
The jars for the packages stored in: /home/jovyan/.ivy2.5.2/jars
org.apache.spark#spark-sql-kafka-0-10_2.13 added as a dependency
org.apache.hadoop#hadoop-aws added as a dependency
com.clickhouse#clickhouse-jdbc added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-c8b001c9-fb2b-4a55-a22b-11d4f82dbe6a;1.0
	confs: [default]
	found org.apache.spark#spark-sql-kafka-0-10_2.13;4.0.2 in central
	found org.apache.spark#spark-token-provider-kafka-0-10_2.13;4.0.2 in central
	found org.apache.kafka#kafka-clients;3.9.1 in central
	found org.lz4#lz4-java;1.8.0 in central
	found org.xerial.snappy#snappy-java;1.1.10.7 in central
	found org.slf4j#slf4j-api;2.0.16 in central
	found org.apache.hadoop#hadoop-client-runtime;3.4.1 in central
	found org.apache.hadoop#had

Spark version: 4.0.2
MinIO endpoint: http://urbangreen-minio:9000
Bucket: staging


In [20]:
sensor_df = (
    spark.read
    .option("recursiveFileLookup", "true")
    .parquet("s3a://staging/raw/postgres/harvests")
    
)
sensor_df.printSchema()
sensor_df.show(5, truncate=False)

root
 |-- id: long (nullable = true)
 |-- farm_id: long (nullable = true)
 |-- crop_id: long (nullable = true)
 |-- quality_grade_id: long (nullable = true)
 |-- weight_kg: decimal(4,3) (nullable = true)
 |-- created_at: long (nullable = true)
 |-- updated_at: long (nullable = true)

+------+-------+-------+----------------+---------+----------+----------+
|id    |farm_id|crop_id|quality_grade_id|weight_kg|created_at|updated_at|
+------+-------+-------+----------------+---------+----------+----------+
|600716|44     |6      |3               |0.797    |1768587540|1768587540|
|601446|44     |7      |1               |1.539    |1768568520|1768568520|
|602176|44     |8      |1               |1.582    |1768587780|1768587780|
|602906|44     |9      |2               |1.439    |1768526400|1768526400|
|603636|44     |10     |1               |1.316    |1768547520|1768547520|
+------+-------+-------+----------------+---------+----------+----------+
only showing top 5 rows


In [12]:
crops_df = (
    spark.read
    .option("recursiveFileLookup", "true")
    .parquet("s3a://staging/raw/postgres/sensors/")
)

crops_df.printSchema()
crops_df.show(5, truncate=False)
    

root
 |-- id: long (nullable = true)
 |-- farm_id: long (nullable = true)
 |-- sensor_type_id: long (nullable = true)
 |-- serial_number: string (nullable = true)
 |-- status: string (nullable = true)
 |-- installed_at: long (nullable = true)
 |-- created_at: long (nullable = true)
 |-- updated_at: long (nullable = true)

+---+-------+--------------+------------------+------+------------+----------+----------+
|id |farm_id|sensor_type_id|serial_number     |status|installed_at|created_at|updated_at|
+---+-------+--------------+------------------+------+------------+----------+----------+
|1  |1      |1             |TEMP-20240315-001 |ACTIVE|1749350000  |1784748802|1784748802|
|2  |1      |2             |HUM-20240322-002  |ACTIVE|1749350000  |1784748802|1784748802|
|3  |1      |3             |LIGHT-20240320-003|ACTIVE|1749350000  |1784748802|1784748802|
|4  |1      |4             |PH-20240318-004   |ACTIVE|1749350000  |1784748802|1784748802|
|5  |1      |5             |ENER-20240325-005 

In [10]:
crop_categories_df = (
    spark.read
    .option("recursiveFileLookup", "true")
    .parquet("s3a://staging/raw/postgres/growing_system_types/")
)
 
crop_categories_df.printSchema()
crop_categories_df.show(5, truncate=False)

root
 |-- id: long (nullable = true)
 |-- name: string (nullable = true)
 |-- description: string (nullable = true)
 |-- created_at: long (nullable = true)
 |-- updated_at: long (nullable = true)

+---+----------------+----------------------------------------------------------------------------------------------------------------+----------+----------+
|id |name            |description                                                                                                     |created_at|updated_at|
+---+----------------+----------------------------------------------------------------------------------------------------------------+----------+----------+
|1  |Vertical Farming|Plants grown in vertically stacked layers with artificial LED lighting and precise nutrient delivery systems    |1784718293|1784718293|
|2  |Tower Farming   |Cylindrical or tower-shaped growing structures that maximize vertical space while minimizing floor footprint    |1784718293|1784718293|
|3  |Flat Bed

In [5]:
from pyspark.sql.functions import col, current_timestamp


dim_crop_df = (
    crops_df
    .join(
        crop_categories_df,
        crops_df.category_id == crop_categories_df.id,
        "left"
    )
    .select(
        crops_df.id.cast("long").alias("crop_id"),
        crops_df.name.alias("name"),
        crops_df.description.alias("description"),

        crops_df.category_id.cast("long").alias("crop_category_id"),
        crop_categories_df.name.alias("category_name"),

        current_timestamp().alias("_loaded_at")
    )
)

In [6]:
dim_crop_df

DataFrame[crop_id: bigint, name: string, description: string, crop_category_id: bigint, category_name: string, _loaded_at: timestamp]

In [7]:
dim_crop_df.printSchema()
dim_crop_df.show(truncate=False)

root
 |-- crop_id: long (nullable = true)
 |-- name: string (nullable = true)
 |-- description: string (nullable = true)
 |-- crop_category_id: long (nullable = true)
 |-- category_name: string (nullable = true)
 |-- _loaded_at: timestamp (nullable = false)

+-------+------------------+---------------------------------------------------------------------------------------------------------------+----------------+---------------+--------------------------+
|crop_id|name              |description                                                                                                    |crop_category_id|category_name  |_loaded_at                |
+-------+------------------+---------------------------------------------------------------------------------------------------------------+----------------+---------------+--------------------------+
|1      |Basil             |Aromatic herb used in Mediterranean cuisine                                                                   

In [8]:
CLICKHOUSE_HOST = os.getenv("CLICKHOUSE_HOST", "urbangreen-clickhouse")
CLICKHOUSE_PORT = os.getenv("CLICKHOUSE_HTTP_PORT", "8123")
CLICKHOUSE_DATABASE = os.getenv("CLICKHOUSE_DB", "urbangreen_dw")
CLICKHOUSE_USER = os.getenv("CLICKHOUSE_USER", "urbangreen")
CLICKHOUSE_PASSWORD = os.getenv("CLICKHOUSE_PASSWORD", "")

In [9]:
dim_crop_df.write \
    .format("jdbc") \
    .option(
        "url",
        "jdbc:clickhouse://urbangreen-clickhouse:8123/urbangreen_dw"
    ) \
    .option("dbtable", "dim_crop") \
    .option("user", "urbangreen") \
    .option("password", "urbangreen-password") \
    .option(
        "driver",
        "com.clickhouse.jdbc.ClickHouseDriver"
    ) \
    .mode("append") \
    .save()

26/07/21 15:42:15 WARN JdbcUtils: Requested isolation level 1, but transactions are unsupported


In [10]:
(
    spark.read.format("jdbc")
    .option("url", "jdbc:clickhouse://urbangreen-clickhouse:8123/urbangreen_dw")
    .option(
        "query",
        """
        SELECT
            job_name,
            cursor_json,
            last_success_at,
            _version
        FROM warehouse_load_state
        """
    )
    .option("user", "urbangreen")
    .option("password", "urbangreen-password")
    .option("driver", "com.clickhouse.jdbc.ClickHouseDriver")
    .load()
    .show()
)

+--------+--------------------+--------------------+-------------+
|job_name|         cursor_json|     last_success_at|     _version|
+--------+--------------------+--------------------+-------------+
|dim_crop|{"last_batch": "2...|2026-07-21 15:38:...|1784648337162|
+--------+--------------------+--------------------+-------------+



In [11]:
from pyspark.sql.functions import input_file_name, regexp_extract
import json
from pyspark.sql.functions import col, current_timestamp

import time
import uuid

from datetime import datetime, timezone

In [20]:
def _read_clickhouse_table(spark, table_name):
    return (
        spark.read.format("jdbc")
        .option("url", "jdbc:clickhouse://urbangreen-clickhouse:8123/urbangreen_dw")
        .option("dbtable", table_name)
        .option("user", "urbangreen")
        .option("password", "urbangreen-password")
        .option("driver", "com.clickhouse.jdbc.ClickHouseDriver")
        .load()
    )

In [21]:
def get_load_state(spark):
    return (
        spark.read.format("jdbc")
        .option("url", "jdbc:clickhouse://urbangreen-clickhouse:8123/urbangreen_dw")
        .option(
            "query",
            """
            SELECT
                job_name,
                cursor_json,
                last_success_at,
                _version
            FROM warehouse_load_state
            """
        )
        .option("user", "urbangreen")
        .option("password", "urbangreen-password")
        .option("driver", "com.clickhouse.jdbc.ClickHouseDriver")
        .load()
    )

In [22]:
def get_cursor(spark, job_name):
    state_df = (
        get_load_state(spark)
        .filter(f"job_name = '{job_name}'")
        .orderBy("_version")
    )

    row = state_df.first()

    if row is None:
        return None

    return json.loads(row["cursor_json"])

In [23]:
def save_cursor(spark, job_name, cursor):
    version = int(time.time() * 1000)

    data = [
        (
            job_name,
            json.dumps(cursor),
            datetime.now(timezone.utc),
            str(uuid.uuid4()),
            version,
        )
    ]

    cursor_df = spark.createDataFrame(
        data,
        [
            "job_name",
            "cursor_json",
            "last_success_at",
            "run_key",
            "_version",
        ],
    )

    (
        cursor_df.write.format("jdbc")
        .option("url", "jdbc:clickhouse://urbangreen-clickhouse:8123/urbangreen_dw")
        .option("dbtable", "warehouse_load_state")
        .option("user", "urbangreen")
        .option("password", "urbangreen-password")
        .option("driver", "com.clickhouse.jdbc.ClickHouseDriver")
        .mode("append")
        .save()
    )

In [24]:
def read_raw_with_batch(spark, path):
    return (
        spark.read.option("recursiveFileLookup", "true")
        .parquet(path)
        .withColumn("_source_file", input_file_name())
        .withColumn(
            "_batch_end", regexp_extract("_source_file", r"__(\d{8}T\d{6}Z)/", 1)
        )
    )

In [25]:
def write_to_clickhouse(df, table_name, mode="append"):

    (
        dim_crop_df.write \
        .format("jdbc") \
        .option(
            "url",
            "jdbc:clickhouse://urbangreen-clickhouse:8123/urbangreen_dw"
        ) \
        .option("dbtable", "dim_crop") \
        .option("user", "urbangreen") \
        .option("password", "urbangreen-password") \
        .option(
            "driver",
            "com.clickhouse.jdbc.ClickHouseDriver"
        ) \
        .mode("append") \
        .save()
    )

In [26]:
from pyspark.sql.functions import col, current_timestamp


def create_dim_crop1(spark):
    cursor = get_cursor(
        spark,
        "dim_crop"
    )

    last_batch = (
        cursor.get("last_batch", "00000000T000000Z")
        if cursor
        else "00000000T000000Z"
    )

    crops_df = (
        read_raw_with_batch(
            spark,
            "s3a://staging/raw/postgres/crops/"
        )
        .filter(
            col("_batch_end") > last_batch
        )
    )

    crop_categories_df = (
        read_raw_with_batch(
            spark,
            "s3a://staging/raw/postgres/crop_categories/"
        )
        .filter(
            col("_batch_end") > last_batch
        )
    )

    dim_crop_df = (
        crops_df
        .join(
            crop_categories_df,
            crops_df.category_id == crop_categories_df.id,
            "left"
        )
        .select(
            crops_df.id.cast("long").alias("crop_id"),
            crops_df.name.alias("name"),
            crops_df.description.alias("description"),
            crops_df.category_id.cast("long").alias("crop_category_id"),
            crop_categories_df.name.alias("category_name"),
            current_timestamp().alias("_loaded_at"),
        )
    )

    latest_batch_row = (
        crops_df
        .select("_batch_end")
        .orderBy(col("_batch_end").desc())
        .first()
    )

    latest_batch = (
        latest_batch_row["_batch_end"]
        if latest_batch_row
        else None
    )

    return dim_crop_df, latest_batch

In [27]:

dim_crop_df, latest_batch = create_dim_crop1(spark)

if latest_batch is not None:
    write_to_clickhouse(
        dim_crop_df,
        "dim_crop"
    )
    
    save_cursor(
        spark,
        "dim_crop",
        {
            "last_batch": latest_batch
        }
    )
